# 01 - Prepare NBA inputs

First step of the pipeline. For one ancestry, this turns the curated clinical
dataset into the covariate, phenotype and keep-sample files the LongGWAS
pipeline reads, then subsets the release's imputed genotypes to those samples
and uploads everything to the working bucket.

Run it once per `ANCESTRY` x `ANALYSIS` combination, then move on to
`02_prep_inputs_wgs.ipynb` (WGS-only samples) and
`03_prep_inputs_non_nba.ipynb` (cohorts outside the release).

Every path comes from the environment - see `.env.template` and the README.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from gwas import config as cfg
from gwas import prep

pd.options.mode.chained_assignment = None

cfg.describe()

In [ ]:
# Mount the release and working buckets with gcsfuse (run once per VM).
for cmd in cfg.mount_commands():
    print(cmd)

## Parameters

In [ ]:
ANALYSIS = 'CS_disc'   # 'CS_disc' (reaching HY >= 3) or 'SURV' (time to HY >= 3)
ANCESTRY = 'EUR'       # AAC, AFR, AJ, AMR, EAS, EUR, MDE, SAS

OUT_DIR = cfg.prep_path(*cfg.prep_parts(ancestry=ANCESTRY), ANALYSIS)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(OUT_DIR)

## Clinical data and master key

The curated datasets are built by the clinical data team; the release master
key supplies ancestry labels, phenotype and study type.

In [ ]:
clinical = pd.read_csv(cfg.clinical_path(ANALYSIS), low_memory=False)
clinical = clinical.rename(columns={'id': 'GP2ID'}).drop_duplicates()

print(f'{len(clinical)} rows, {clinical.GP2ID.nunique()} samples')
display(clinical.cohort.value_counts())
display(clinical.head())

In [ ]:
master = pd.read_csv(cfg.master_key())

in_master = master[master.GP2ID.isin(clinical.GP2ID)]
print(f'{len(in_master)} of {clinical.GP2ID.nunique()} samples found in the master key')
display(in_master.nba_label.value_counts())
display(in_master.nba.value_counts(dropna=False))

# Samples available prior to GP2 release
missing = clinical[~clinical.GP2ID.isin(master.GP2ID)]
if len(missing):
    print(f'\n{missing.GP2ID.nunique()} samples missing from the master key:')
    display(missing.cohort.value_counts())

## Select the ancestry

In [ ]:
key_cols = ['GP2ID', 'study_type', 'GP2_phenotype', 'nba_label']
full_data = clinical.merge(master[key_cols], on='GP2ID', how='inner')

subset = full_data[full_data.nba_label == ANCESTRY]

# Only case/control designs are modelled
display(subset.study_type.value_counts())
subset = subset[subset.study_type == 'Case(/Control)']

subset = prep.check_required_fields(subset, ANALYSIS)
print(f'{subset.GP2ID.nunique()} samples in {ANCESTRY}')
display(subset.cohort.value_counts())

## Keep-sample list and PLINK IDs

The imputed `.psam` supplies the `#FID` / `IID` pair the pipeline matches on,
and drops any clinical sample that was not genotyped.

In [ ]:
psam = pd.read_csv(f'{cfg.imputed_pfile(ANCESTRY, 22)}.psam', sep='\t')

keep = psam[psam.IID.isin(subset.GP2ID)]
print(f'{len(keep)} of {subset.GP2ID.nunique()} samples have imputed genotypes')
keep[['#FID', 'IID']].to_csv(OUT_DIR / f'{ANCESTRY}_keep_samples.txt',
                             sep='\t', index=False)

subset = subset.merge(psam[['#FID', 'IID']], left_on='GP2ID', right_on='IID', how='inner')
display(subset.head())

## Covariates and phenotypes

Covariate files need PLINK-style `#FID` and `IID` columns plus `PHENO`

In [ ]:
covar = prep.format_covariates(subset, ANALYSIS)
covar.to_csv(OUT_DIR / cfg.covar_file(ANALYSIS, ANCESTRY), sep='\t', index=False)

print(f'{len(covar)} covariate rows')
display(covar.head())

# Negative disease duration means onset is recorded after baseline - worth a
# look before the GWAS runs.
bad_dd = covar[covar.disease_duration < 0]
if len(bad_dd):
    print(f'WARNING: {len(bad_dd)} samples have a negative disease duration')
    display(bad_dd)

In [ ]:
pheno = prep.format_phenotypes(subset, ANALYSIS)
pheno.to_csv(OUT_DIR / cfg.pheno_file(ANALYSIS, ANCESTRY), sep='\t', index=False)
display(pheno.head())

## Survival-only fixes

Two adjustments the Cox model needs: an event at day 0 carries no follow-up,
and a cohort too small to fit its own stratum destabilises the per-cohort fit.
Both files are rewritten in place, after a backup copy.

In [ ]:
if ANALYSIS == 'SURV':
    covar_path = OUT_DIR / cfg.covar_file(ANALYSIS, ANCESTRY)
    pheno_path = OUT_DIR / cfg.pheno_file(ANALYSIS, ANCESTRY)

    for path in (covar_path, pheno_path):
        backup = path.with_name(path.stem + '_PRE_FIX.tsv')
        backup.write_bytes(path.read_bytes())
        print(f'backed up {backup.name}')

    covar = pd.read_csv(covar_path, sep='\t')
    pheno = pd.read_csv(pheno_path, sep='\t')

    covar, pheno = prep.drop_zero_time_events(covar, pheno)
    covar, pheno = prep.drop_small_cohorts(covar, pheno, min_n=50)

    covar.to_csv(covar_path, sep='\t', index=False)
    pheno.to_csv(pheno_path, sep='\t', index=False)

    display(covar.cohort.value_counts())
    display(pheno.y.value_counts())

## Genotype subsetting

The same genotype subset serves every analysis for this ancestry, so the
keep-list is the union over the analyses that have been prepared.

In [ ]:
ANALYSES = ['CS_disc', 'SURV']

base_dir = cfg.prep_path(*cfg.prep_parts(ancestry=ANCESTRY))
plink_dir = base_dir / 'PLINK'
plink_dir.mkdir(parents=True, exist_ok=True)

keep_all = prep.keep_samples_across_analyses(base_dir, ANALYSES, ANCESTRY)
keep_file = plink_dir / f'{ANCESTRY}_keep_samples.txt'
keep_all.to_csv(keep_file, sep='\t', index=False)

print(f'{len(keep_all)} unique samples across {ANALYSES}')

# Nothing should be missing; if it is, an analysis was prepared after the
# genotypes were subset.
for analysis in ANALYSES:
    path = base_dir / analysis / cfg.covar_file(analysis, ANCESTRY)
    if path.exists():
        check = pd.read_csv(path, sep='\t')
        dropped = check[~check.IID.isin(keep_all.IID)]
        print(f'{analysis}: {len(dropped)} samples not in the keep list')

In [ ]:
# One bash script, run once per chromosome, that subsets the release pfile
# and copies the result to the bucket. ${CHROM} is filled in by the script.
scratch = cfg.TMP_DIR / 'hold_genotypes'
scratch.mkdir(parents=True, exist_ok=True)

script = prep.write_subset_script(
    cfg.WORK_DIR / 'subset_scripts' / ANCESTRY / f'{ANCESTRY}_subset_plink.sh',
    pfile=cfg.imputed_pfile(ANCESTRY),
    keep_file=keep_file,
    out_prefix=str(scratch / f'chr${{CHROM}}_hy_subsetted_{ANCESTRY}'),
    dest=cfg.prep_url(*cfg.prep_parts(ancestry=ANCESTRY), 'PLINK'),
)
print(f'\ntest on one chromosome first:\n{script} 22')

In [ ]:
for chrom in range(1, 23):
    cfg.shell_do(f'{script} {chrom}', log=True)

## Upload the covariate and phenotype files

In [ ]:
dest = cfg.prep_url(*cfg.prep_parts(ancestry=ANCESTRY))
print(f'gsutil -m cp -r {OUT_DIR} {dest}/')

Next: `02_prep_inputs_wgs.ipynb` and `03_prep_inputs_non_nba.ipynb` for the
remaining datasets, then `04_launch_gwas.ipynb`.